In [ ]:
# ── Imports ───────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller
from prophet import Prophet
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
print('Libraries loaded.')

In [ ]:
# Load dataset (place household_power_consumption.txt in working directory)
df_raw = pd.read_csv(
    'household_power_consumption.txt',
    sep=';',
    parse_dates={'datetime': ['Date', 'Time']},
    dayfirst=True,
    na_values=['?'],
    low_memory=False
)
print(f'Shape: {df_raw.shape}')
print(f'Date range: {df_raw.datetime.min()} → {df_raw.datetime.max()}')
df_raw.head()

## 2. Data Cleaning & Resampling

In [ ]:
# Set datetime index
df_raw.set_index('datetime', inplace=True)

# Focus on Global_active_power
series = df_raw['Global_active_power'].copy()

# Drop missing values
print(f'Missing before: {series.isna().sum()} ({series.isna().mean():.1%})')
series.dropna(inplace=True)

# Resample to hourly — use mean
df_hourly = series.resample('H').mean().dropna()
print(f'Hourly records: {len(df_hourly)}')
df_hourly.head()

## 3. Feature Engineering

In [ ]:
df_feat = pd.DataFrame({'power': df_hourly})

# Time-based features
df_feat['hour']       = df_feat.index.hour
df_feat['dayofweek']  = df_feat.index.dayofweek
df_feat['month']      = df_feat.index.month
df_feat['is_weekend'] = (df_feat['dayofweek'] >= 5).astype(int)
df_feat['quarter']    = df_feat.index.quarter

# Lag features
df_feat['lag_1h']  = df_feat['power'].shift(1)
df_feat['lag_24h'] = df_feat['power'].shift(24)
df_feat['lag_168h']= df_feat['power'].shift(168)  # 1 week

# Rolling statistics
df_feat['rolling_mean_24h'] = df_feat['power'].shift(1).rolling(24).mean()
df_feat['rolling_std_24h']  = df_feat['power'].shift(1).rolling(24).std()

df_feat.dropna(inplace=True)
print(f'Feature matrix shape: {df_feat.shape}')
df_feat.head()

## 4. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 12))

# Full time series (weekly average for readability)
df_hourly.resample('D').mean().plot(ax=axes[0], color='steelblue', lw=0.8)
axes[0].set_title('Daily Average Global Active Power (Full Series)')
axes[0].set_ylabel('kW')

# Average by hour of day
df_feat.groupby('hour')['power'].mean().plot(kind='bar', ax=axes[1], color='darkorange')
axes[1].set_title('Average Power by Hour of Day')
axes[1].set_xlabel('Hour')
axes[1].set_ylabel('Mean kW')

# Average by day of week
days = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
df_feat.groupby('dayofweek')['power'].mean().plot(kind='bar', ax=axes[2], color='seagreen')
axes[2].set_title('Average Power by Day of Week')
axes[2].set_xticklabels(days, rotation=0)
axes[2].set_ylabel('Mean kW')

plt.tight_layout()
plt.savefig('task3_eda.png', dpi=120)
plt.show()

In [ ]:
# ── Stationarity check ────────────────────────────────────────────────────────
adf = adfuller(df_hourly.dropna())
print(f'ADF Statistic: {adf[0]:.4f}')
print(f'p-value: {adf[1]:.6f}')
print('Series is', 'stationary' if adf[1] < 0.05 else 'non-stationary')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(df_hourly.dropna(), lags=48, ax=axes[0])
plot_pacf(df_hourly.dropna(), lags=48, ax=axes[1])
plt.tight_layout()
plt.savefig('task3_acf_pacf.png', dpi=120)
plt.show()

## 5. Train / Test Split

In [ ]:
# Use last 7 days (168 hours) as test set
TEST_HOURS = 168

train_series = df_hourly[:-TEST_HOURS]
test_series  = df_hourly[-TEST_HOURS:]

df_train = df_feat.iloc[:-TEST_HOURS]
df_test  = df_feat.iloc[-TEST_HOURS:]

print(f'Train: {len(train_series)} hours | Test: {len(test_series)} hours')

## 6. Model 1 — ARIMA

In [ ]:
# Fit ARIMA(2,0,2) — chosen based on ACF/PACF
# For large series, we fit on a recent window to keep training fast
ARIMA_WINDOW = 2000  # recent hours
arima_train = train_series.iloc[-ARIMA_WINDOW:]

arima_model = ARIMA(arima_train, order=(2, 0, 2)).fit()

# Forecast
arima_forecast = arima_model.forecast(steps=TEST_HOURS)
arima_forecast.index = test_series.index

mae_arima  = mean_absolute_error(test_series, arima_forecast)
rmse_arima = np.sqrt(mean_squared_error(test_series, arima_forecast))
print(f'ARIMA  →  MAE: {mae_arima:.4f} | RMSE: {rmse_arima:.4f}')

## 7. Model 2 — Prophet

In [ ]:
# Prophet requires columns 'ds' and 'y'
prophet_train = pd.DataFrame({'ds': train_series.index, 'y': train_series.values})

prophet_model = Prophet(
    daily_seasonality=True,
    weekly_seasonality=True,
    yearly_seasonality=True,
    changepoint_prior_scale=0.05
)
prophet_model.fit(prophet_train)

# Build future dataframe
future = prophet_model.make_future_dataframe(periods=TEST_HOURS, freq='H')
forecast_prop = prophet_model.predict(future)

prophet_pred = forecast_prop.iloc[-TEST_HOURS:]['yhat'].values

mae_prophet  = mean_absolute_error(test_series.values, prophet_pred)
rmse_prophet = np.sqrt(mean_squared_error(test_series.values, prophet_pred))
print(f'Prophet →  MAE: {mae_prophet:.4f} | RMSE: {rmse_prophet:.4f}')

## 8. Model 3 — XGBoost

In [ ]:
FEATURE_COLS = ['hour','dayofweek','month','is_weekend','quarter',
                'lag_1h','lag_24h','lag_168h',
                'rolling_mean_24h','rolling_std_24h']

X_train_xgb = df_train[FEATURE_COLS]
y_train_xgb = df_train['power']
X_test_xgb  = df_test[FEATURE_COLS]
y_test_xgb  = df_test['power']

xgb_model = xgb.XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train_xgb, y_train_xgb,
              eval_set=[(X_test_xgb, y_test_xgb)],
              verbose=False)

xgb_pred = xgb_model.predict(X_test_xgb)

mae_xgb  = mean_absolute_error(y_test_xgb, xgb_pred)
rmse_xgb = np.sqrt(mean_squared_error(y_test_xgb, xgb_pred))
print(f'XGBoost →  MAE: {mae_xgb:.4f} | RMSE: {rmse_xgb:.4f}')

## 9. Model Comparison & Visualizations

In [ ]:
# ── Metrics comparison table ──────────────────────────────────────────────────
results = pd.DataFrame({
    'Model': ['ARIMA', 'Prophet', 'XGBoost'],
    'MAE':   [mae_arima, mae_prophet, mae_xgb],
    'RMSE':  [rmse_arima, rmse_prophet, rmse_xgb]
}).set_index('Model').round(4)

print('\n=== Model Comparison ===')
print(results)
print(f"\nBest MAE:  {results['MAE'].idxmin()}")
print(f"Best RMSE: {results['RMSE'].idxmin()}")

In [ ]:
# ── Actual vs Forecast plots ──────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(16, 14), sharex=False)

plot_data = [
    ('ARIMA',   arima_forecast.values, test_series.index, mae_arima,  rmse_arima,  '#E63946'),
    ('Prophet', prophet_pred,          test_series.index, mae_prophet, rmse_prophet,'#2A9D8F'),
    ('XGBoost', xgb_pred,             y_test_xgb.index,  mae_xgb,    rmse_xgb,    '#E9C46A'),
]

for ax, (name, preds, idx, mae, rmse, color) in zip(axes, plot_data):
    ax.plot(idx, test_series.values, label='Actual',    color='#457B9D', lw=1.5, alpha=0.9)
    ax.plot(idx, preds,             label=f'{name} Forecast', color=color, lw=1.5, linestyle='--')
    ax.set_title(f'{name} — MAE: {mae:.4f} | RMSE: {rmse:.4f}')
    ax.set_ylabel('Global Active Power (kW)')
    ax.legend()
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%m-%d %Hh'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.suptitle('Actual vs Forecasted Energy Consumption (Last 7 Days)', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('task3_forecasts.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# ── Metrics bar chart ─────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

results['MAE'].plot(kind='bar', ax=axes[0], color=['#E63946','#2A9D8F','#E9C46A'], edgecolor='white')
axes[0].set_title('MAE by Model (lower = better)')
axes[0].set_ylabel('MAE')
axes[0].tick_params(axis='x', rotation=0)

results['RMSE'].plot(kind='bar', ax=axes[1], color=['#E63946','#2A9D8F','#E9C46A'], edgecolor='white')
axes[1].set_title('RMSE by Model (lower = better)')
axes[1].set_ylabel('RMSE')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
plt.savefig('task3_metrics.png', dpi=120)
plt.show()

In [ ]:
# ── XGBoost Feature Importance ────────────────────────────────────────────────
feat_importance = pd.Series(xgb_model.feature_importances_, index=FEATURE_COLS).sort_values()
feat_importance.plot(kind='barh', figsize=(10, 5), color='steelblue', edgecolor='white')
plt.title('XGBoost Feature Importance')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.savefig('task3_xgb_features.png', dpi=120)
plt.show()